# Data Leakage:
* Data leakage is when information from outside the training data — specifically, information the model shouldn't have access to when making a real prediction — accidentally sneaks into the training process. The result: our model looks great during evaluation (great MAE, high accuracy) but performs much worse in the real world, because it learned to "cheat" using information it won't actually have at prediction time.
* It's one of the most dangerous ML bugs because it doesn't throw an error — it silently makes our model look better than it actually is. This connects directly to bugs we have already hit this session, even though we didn't call them "leakage" at the time.
* Note: `Data leakage = our evaluation number is lying to us. The model looks better than it actually is, because somewhere, some piece of information it shouldn't have had access to yet — either a feature that only exists after the fact, or a peek at the "hidden" test data — snuck in during training.`
### There are two main types of leakage:
#### Target leakage:
`A feature only exists because the answer already happened — it secretly contains the answer in disguise.
Example: predicting rain using "did people carry umbrellas" — umbrellas come after rain starts, so this feature won't exist yet when you actually need a prediction.
Test: would this feature's value be known before the outcome happens? If not → target leakage.`
#### Train-Test Contamination:
`Validation/test data accidentally influences how training data is prepared — so the "unseen" test isn't really unseen anymore.
Example: fitting an imputer (mean/mode) on train + valid combined, instead of train only — validation data quietly shapes its own preprocessing.
Fix: always fit() only on train, transform() (never re-fit) on valid/test.`

In this example, we will learn one way to detect and remove target leakage.

We will use a dataset about credit card applications. The end result is that information about each credit card application is stored in a DataFrame X. We'll use it to predict which applications were accepted in a Series y

In [25]:
import pandas as pd
# Set the maximum number of rows to unlimited
pd.set_option('display.max_rows', None)

data=pd.read_csv('AER_credit_card_data.csv',
                true_values=['yes'],false_values=['no']) #This tells pandas: "wherever you see the exact text 'yes' in the file,
#treat it as boolean True. Wherever you see 'no', treat it as boolean False."

data.dropna(subset=['card'],axis=0,inplace=True)
y=data.card
X=data.drop(['card'],axis=1)
data[data['card']==True]

,card,reports,age,income,share,expenditure,owner,selfemp,dependents,months,majorcards,active
0,True,0,37.666670,4.5200,0.033270,124.983300,True,False,3,54,1,12
1,True,0,33.250000,2.4200,0.005217,9.854167,False,False,3,34,1,13
2,True,0,33.666670,4.5000,0.004156,15.000000,True,False,4,58,1,5
3,True,0,30.500000,2.5400,0.065214,137.869200,False,False,0,25,1,7
4,True,0,32.166670,9.7867,0.067051,546.503300,True,False,2,64,1,5
5,True,0,23.250000,2.5000,0.044438,91.996670,False,False,0,54,1,1
6,True,0,27.916670,3.9600,0.012576,40.833330,False,False,2,7,1,5
7,True,0,29.166670,2.3700,0.076434,150.790000,True,False,0,77,1,3
8,True,0,37.000000,3.8000,0.245628,777.821700,True,False,0,97,1,6
9,True,0,28.416670,3.2000,0.019780,52.580000,False,False,0,65,1,18


In [26]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

model=RandomForestClassifier(n_estimators=100,random_state=0)
my_pipeline=Pipeline(steps=[
    ('model',model)
])

validation_scores=cross_val_score(
    my_pipeline,
    X,y,
    cv=5,
    scoring='accuracy'
)

print(f"Cross-Validation Accuracy: {round(validation_scores.mean()*100,2)}%")

Cross-Validation Accuracy: 98.03%


we'll find that it's very rare to find models that are accurate 98% of the time. It happens, but it's uncommon enough that we should inspect the data more closely for target leakage.

Here is a summary of the data:

* `card`: 1 if credit card application accepted, 0 if not
* `reports`: Number of major derogatory reports
* `age`: Age n years plus twelfths of a year
* `income`: Yearly income (divided by 10,000)
* `share`: Ratio of monthly credit card expenditure to yearly income
* `expenditure`: Average monthly credit card expenditure
* `owner`: 1 if owns home, 0 if rents
* `selfempl`: 1 if self-employed, 0 if not
* `dependents`: 1 + number of dependents
* `months`: Months living at current address
* `majorcards`: Number of major credit cards held
* `active`: Number of active credit accounts

A few variables look suspicious. For example, does `expenditure` mean expenditure on this card or on cards used before applying?

At this point, basic data comparisons can be very helpful:

In [27]:
expenditure_cardholders=X.expenditure[y]
expenditure_non_cardholders=X.expenditure[~y]

print(f'Fraction of those who received a card and had no expenditures: {round((expenditure_cardholders==0).mean()*100,2)}%')
print(f'Fraction of those who did not receive a card and had no expenditures: {round((expenditure_non_cardholders==0).mean()*100,2)}%')

Fraction of those who received a card and had no expenditures: 2.05%
Fraction of those who did not receive a card and had no expenditures: 100.0%




As shown above, everyone who did not receive a card had no expenditures, while only 2% of those who received a card had no expenditures. It's not surprising that our model appeared to have a high accuracy. But this also seems to be a case of target leakage, where expenditures probably means expenditures on the card they applied for.

Since `share` is partially determined by `expenditure`, it should be excluded too. The variables `active` and `majorcards` are a little less clear, but from the description, they sound concerning. In most situations, it's better to be safe than sorry if you can't track down the people who created the data to find out more.

In [28]:
# Drop leaky predictors from dataset
leaks=['expenditure', 'share', 'active', 'majorcards']
X2=X.drop(leaks,axis=1)

validation_scores=cross_val_score(
    my_pipeline,
    X2,y,
    cv=5,
    scoring='accuracy'
)

print(f"Cross-Validation Accuracy: {round(validation_scores.mean()*100,2)}%")

Cross-Validation Accuracy: 82.94%


Data leakage can be multi-million dollar mistake in many data science applications. Careful separation of training and validation data can prevent train-test contamination, and pipelines can help implement this separation. Likewise, a combination of caution, common sense, and data exploration can help identify target leakage.